<a href="https://colab.research.google.com/github/jinbaaaaaang/huwari/blob/main/huwari_attr_train.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# 메타데이터 다시 파싱 (이미지 식별자 기반)
import zipfile
import json
import pandas as pd
from collections import defaultdict
import random

LABEL_ZIP = "/content/drive/MyDrive/codimodi/01_data/raw/aihub_kfashion/라벨링데이터.zip"
SAVE_DIR  = "/content/drive/MyDrive/codimodi/01_data/processed"

STYLE_MAP_ORIGINAL = {
    "로맨틱":          "로맨틱",
    "페미닌":          "로맨틱",
    "스트리트":        "스트리트",
    "힙합":           "스트리트",
    "펑크":           "스트리트",
    "스포티":          "스포티",
    "리조트":          "캐주얼",
    "모던":            "미니멀",
    "소피스트케이티드":  "포멀",
    "클래식":          "포멀",
    "매니시":          "긱시크",
    "톰보이":          "긱시크",
    "젠더리스":        "긱시크",
    "레트로":          "빈티지",
    "히피":            "빈티지",
    "오리엔탈":        "빈티지",
    "밀리터리":        "고프코어",
    "컨트리":          "고프코어",
    "웨스턴":          "고프코어",
    "프레피":          "Y2K",
    "키치":            "Y2K",
    "아방가르드":       "Y2K",
    "섹시":            "로맨틱",
    "기타":            None,
}

# 원본 스타일 → zip 매핑
ORIGINAL_STYLE_TO_ZIP_FOLDER = {
    "로맨틱":          ("원천데이터_1.zip", "로맨틱"),
    "모던":            ("원천데이터_1.zip", "모던"),
    "리조트":          ("원천데이터_1.zip", "리조트"),
    "레트로":          ("원천데이터_1.zip", "레트로"),
    "매니시":          ("원천데이터_1.zip", "매니시"),
    "밀리터리":        ("원천데이터_1.zip", "밀리터리"),
    "섹시":            ("원천데이터_1.zip", "섹시"),
    "소피스트케이티드":  ("원천데이터_1.zip", "소피스트케이티드"),
    "스트리트":        ("원천데이터_2.zip", "스트리트"),
    "스포티":          ("원천데이터_3.zip", "스포티"),
    "페미닌":          ("원천데이터_3.zip", "페미닌"),
    "클래식":          ("원천데이터_3.zip", "클래식"),
    "컨트리":          ("원천데이터_3.zip", "컨트리"),
    "젠더리스":        ("원천데이터_3.zip", "젠더리스"),
    "톰보이":          ("원천데이터_3.zip", "톰보이"),
    "히피":            ("원천데이터_3.zip", "히피"),
    "키치":            ("원천데이터_3.zip", "키치"),
    "프레피":          ("원천데이터_3.zip", "프레피"),
    "아방가르드":       ("원천데이터_3.zip", "아방가르드"),
    "웨스턴":          ("원천데이터_3.zip", "웨스턴"),
    "힙합":            ("원천데이터_3.zip", "힙합"),
    "펑크":            ("원천데이터_3.zip", "펑크"),
    "오리엔탈":        ("원천데이터_3.zip", "오리엔탈"),
}

TARGET_STYLES  = ["캐주얼", "고프코어", "미니멀", "긱시크", "로맨틱",
                  "빈티지", "포멀", "Y2K", "스트리트", "스포티"]
SAMPLES_PER_STYLE = 5000

def extract_material(labeling):
    for key in ["상의", "아우터", "하의", "원피스"]:
        item = labeling.get(key, [{}])
        if item and item[0]:
            소재 = item[0].get("소재", [])
            if 소재:
                return 소재[0]
    return None

def extract_pattern(labeling):
    for key in ["상의", "아우터", "하의", "원피스"]:
        item = labeling.get(key, [{}])
        if item and item[0]:
            프린트 = item[0].get("프린트", [])
            if 프린트:
                return 프린트[0]
    return None

print("JSON 파싱 중...")

style_to_files = defaultdict(list)
rows = []

with zipfile.ZipFile(LABEL_ZIP, 'r') as zf:
    json_files = [f for f in zf.namelist() if f.endswith('.json')]

    # 스타일별 파일 분류
    for fname in json_files:
        folder = fname.split('/')[0]
        mapped = STYLE_MAP_ORIGINAL.get(folder)
        if mapped:
            style_to_files[mapped].append((fname, folder))

    # 스타일별 샘플링 + 파싱
    for style in TARGET_STYLES:
        files = style_to_files[style]
        n     = min(SAMPLES_PER_STYLE, len(files))
        sampled = random.sample(files, n)

        for fname, original_style in sampled:
            try:
                with zf.open(fname) as f:
                    data = json.load(f)

                # 이미지 식별자 추출
                if "이미지 정보" in data:
                    img_id = data["이미지 정보"].get("이미지 식별자")
                else:
                    img_id = data["데이터셋 정보"].get("파일 번호")

                labeling = data["데이터셋 정보"]["데이터셋 상세설명"]["라벨링"]
                material = extract_material(labeling)
                pattern  = extract_pattern(labeling)

                # zip 파일명 + 경로
                zip_info = ORIGINAL_STYLE_TO_ZIP_FOLDER.get(original_style)
                if zip_info is None:
                    continue
                zip_name, zip_folder = zip_info

                rows.append({
                    "img_id":      img_id,
                    "filename":    f"{img_id}.jpg",
                    "zip_name":    zip_name,
                    "zip_img_path": f"{zip_folder}/{img_id}.jpg",
                    "style":       style,
                    "material":    material,
                    "pattern":     pattern,
                    "original_style": original_style,
                })
            except Exception as e:
                continue

df_new = pd.DataFrame(rows)
print(f"파싱 완료: {len(df_new)}개")
print("\n재질 분포 (상위 10개):")
print(df_new["material"].value_counts().head(10))
print("\n패턴 분포:")
print(df_new["pattern"].value_counts().head(10))
print("\n스타일 분포:")
print(df_new["style"].value_counts())

df_new.to_csv(f"{SAVE_DIR}/metadata_50k_v2.csv", index=False, encoding="utf-8-sig")
print(f"\n저장 완료: metadata_50k_v2.csv")

Mounted at /content/drive
JSON 파싱 중...
파싱 완료: 50000개

재질 분포 (상위 10개):
material
우븐        20784
저지         8936
니트         7182
시폰         2905
울/캐시미어     2346
린넨         2134
데님          926
실크          776
스판덱스        463
레이스         348
Name: count, dtype: int64

패턴 분포:
pattern
무지       30170
체크        2841
레터링       2552
그래픽       2392
플로럴       2366
스트라이프     2272
믹스         941
도트         757
페이즐리       380
깅엄         265
Name: count, dtype: int64

스타일 분포:
style
캐주얼     5000
고프코어    5000
미니멀     5000
긱시크     5000
로맨틱     5000
빈티지     5000
포멀      5000
Y2K     5000
스트리트    5000
스포티     5000
Name: count, dtype: int64

저장 완료: metadata_50k_v2.csv


In [ ]:
# 이미지 제대로 읽히는지 확인
import zipfile
import io
from PIL import Image

df_check = pd.read_csv(f"{SAVE_DIR}/metadata_50k_v2.csv")
row = df_check.iloc[0]

print(f"zip_name: {row['zip_name']}")
print(f"zip_img_path: {row['zip_img_path']}")

zip_path = f"/content/drive/MyDrive/codimodi/01_data/raw/aihub_kfashion/{row['zip_name']}"
with zipfile.ZipFile(zip_path, 'r') as zf:
    all_files = zf.namelist()
    print(f"\n경로 존재: {row['zip_img_path'] in all_files}")

    # 비슷한 파일 찾기
    img_id = str(row['img_id'])
    similar = [f for f in all_files if img_id in f]
    print(f"비슷한 파일: {similar[:5]}")

zip_name: 원천데이터_1.zip
zip_img_path: 리조트/1243724.jpg

경로 존재: True
비슷한 파일: ['리조트/1243724.jpg']


In [ ]:
import zipfile

# 각 zip의 폴더 목록 확인
for zip_name in ["원천데이터_1.zip", "원천데이터_2.zip", "원천데이터_3.zip"]:
    zip_path = f"/content/drive/MyDrive/codimodi/01_data/raw/aihub_kfashion/{zip_name}"
    with zipfile.ZipFile(zip_path, 'r') as zf:
        files = zf.namelist()
        folders = set(f.split('/')[0] for f in files if '/' in f)
        print(f"\n{zip_name}")
        print(f"총 파일 수: {len(files)}")
        print(f"폴더 목록: {folders}")


원천데이터_1.zip
총 파일 수: 298612
폴더 목록: {'매니시', '레트로', '섹시', '리조트', '모던', '로맨틱', '밀리터리', '소피스트케이티드', '기타'}

원천데이터_2.zip
총 파일 수: 449495
폴더 목록: {'스트리트'}

원천데이터_3.zip
총 파일 수: 219723
폴더 목록: {'힙합', '스포티', '키치', '펑크', '히피', '프레피', '페미닌', '클래식', '웨스턴', '오리엔탈', '아방가르드', '톰보이', '컨트리', '젠더리스'}


## 설치

In [ ]:
!pip install timm --quiet

## 라이브러리 임포트

In [ ]:
import os
import json
import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm
from collections import Counter

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import timm

from google.colab import drive
drive.mount('/content/drive')

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"디바이스: {device}")

DRIVE_ROOT  = "/content/drive/MyDrive/codimodi/01_data/processed"
META_PATH   = f"{DRIVE_ROOT}/metadata_masked.csv"
IMAGE_ROOT  = f"{DRIVE_ROOT}/images"
SAVE_DIR    = "/content/drive/MyDrive/fashion_harmony"
os.makedirs(SAVE_DIR, exist_ok=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
디바이스: cuda


## FashionHarmony 클래스 체계 정의

In [ ]:
CATEGORY_CLASSES = ["상의", "하의", "신발", "모자", "악세서리"]

MATERIAL_CLASSES = [
    "데님", "니트", "실크", "가죽", "울", "면", "패딩", "기타"
]

PATTERN_CLASSES = [
    "무지", "스트라이프", "체크", "도트", "플로럴",
    "그래픽", "호피·뱀피", "카무플라쥬", "기타"
]

STYLE_CLASSES = [
    "캐주얼", "고프코어", "미니멀", "긱시크", "로맨틱",
    "빈티지", "포멀", "Y2K", "스트리트", "스포티"
]

# K-Fashion → FashionHarmony 매핑
STYLE_MAP = {
    "Casual":   "캐주얼",
    "Gorpcore": "고프코어",
    "Minimal":  "미니멀",
    "GeekChic": "긱시크",
    "Romantic": "로맨틱",
    "Vintage":  "빈티지",
    "Formal":   "포멀",
    "Y2K":      "Y2K",
    "스트리트":  "스트리트",
    "스포티":   "스포티",
}

def map_material(label: str) -> str:
    if pd.isna(label) or str(label) == "-1":
        return None
    label = str(label).strip()
    if "데님" in label:
        return "데님"
    if any(x in label for x in ["니트", "저지", "헤어 니트"]):
        return "니트"
    if any(x in label for x in ["실크", "시폰"]):
        return "실크"
    if any(x in label for x in ["가죽", "스웨이드", "무스탕"]):
        return "가죽"
    if any(x in label for x in ["울/캐시미어", "울", "트위드", "코듀로이"]):
        return "울"
    if any(x in label for x in ["린넨", "우븐"]):
        return "면"
    if any(x in label for x in ["패딩", "플리스", "퍼"]):
        return "패딩"
    # 벨벳/메시/레이스/스판덱스/네오프렌 등 → 기타
    return "기타"

def map_pattern(label: str) -> str:
    if pd.isna(label) or str(label) == "-1":
        return None
    label = str(label).strip()
    if "무지" in label and "그래픽" not in label and "레터링" not in label:
        return "무지"
    if "스트라이프" in label:
        return "스트라이프"
    if any(x in label for x in ["체크", "깅엄", "하운즈투스"]):
        return "체크"
    if "도트" in label:
        return "도트"
    if "플로럴" in label:
        return "플로럴"
    if any(x in label for x in ["그래픽", "레터링"]):
        return "그래픽"
    if any(x in label for x in ["호피", "뱀피"]):
        return "호피·뱀피"
    if "카무플라쥬" in label:
        return "카무플라쥬"
    # 아가일/타이다이/그라데이션/페이즐리/기타 → 기타로 통합
    return "기타"

print("클래스 정의 완료")
print(f"재질: {len(MATERIAL_CLASSES)}개")
print(f"패턴: {len(PATTERN_CLASSES)}개")
print(f"스타일: {len(STYLE_CLASSES)}개")

클래스 정의 완료
재질: 8개
패턴: 9개
스타일: 10개


## 데이터 로드 + 매핑

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

SAVE_DIR = "/content/drive/MyDrive/codimodi/01_data/processed"

df = pd.read_csv(f"{SAVE_DIR}/metadata_50k_v2.csv")
print(f"전체 샘플 수: {len(df)}")

# FashionHarmony 클래스로 매핑
df["harmony_material"] = df["material"].apply(map_material)
df["harmony_pattern"]  = df["pattern"].apply(map_pattern)

df["h_material_id"] = df["harmony_material"].apply(
    lambda x: MATERIAL_CLASSES.index(x) if x in MATERIAL_CLASSES else -1
)
df["h_pattern_id"] = df["harmony_pattern"].apply(
    lambda x: PATTERN_CLASSES.index(x) if x in PATTERN_CLASSES else -1
)
df["h_style_id"] = df["style"].apply(
    lambda x: STYLE_CLASSES.index(x) if x in STYLE_CLASSES else -1
)

# style 없는 샘플 제거
df = df[df["h_style_id"] != -1].reset_index(drop=True)
print(f"유효한 샘플: {len(df)}")

print("\n패턴 분포:")
print(df["harmony_pattern"].value_counts())
print("\n재질 분포:")
print(df["harmony_material"].value_counts())
print("\n스타일 분포:")
print(df["style"].value_counts())

# train/val split
train_df, val_df = train_test_split(
    df, test_size=0.15,
    random_state=42, stratify=df["h_style_id"]
)
train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)
print(f"\ntrain: {len(train_df)}, val: {len(val_df)}")

전체 샘플 수: 50000
유효한 샘플: 50000

패턴 분포:
harmony_pattern
무지       30170
그래픽       4944
체크        3106
플로럴       2366
스트라이프     2272
기타        1771
도트         757
호피·뱀피      140
카무플라쥬       59
Name: count, dtype: int64

재질 분포:
harmony_material
면     22918
니트    16257
실크     3681
울      2759
기타     1142
데님      926
패딩      782
가죽      232
Name: count, dtype: int64

스타일 분포:
style
캐주얼     5000
고프코어    5000
미니멀     5000
긱시크     5000
로맨틱     5000
빈티지     5000
포멀      5000
Y2K     5000
스트리트    5000
스포티     5000
Name: count, dtype: int64

train: 42500, val: 7500


In [ ]:
import zipfile
import os
from tqdm.auto import tqdm

EXTRACT_DIR = "/content/kfashion_images"
os.makedirs(EXTRACT_DIR, exist_ok=True)
ZIP_ROOT = "/content/drive/MyDrive/codimodi/01_data/raw/aihub_kfashion"

all_df = pd.concat([train_df, val_df], ignore_index=True)
zip_groups = all_df.groupby("zip_name")

for zip_name, group in zip_groups:
    zip_path = f"{ZIP_ROOT}/{zip_name}"
    print(f"{zip_name}: {len(group)}개 추출 중...")
    with zipfile.ZipFile(zip_path, 'r') as zf:
        for _, row in tqdm(group.iterrows(), total=len(group)):
            try:
                out_path = f"{EXTRACT_DIR}/{row['img_id']}.jpg"
                if not os.path.exists(out_path):
                    img_data = zf.read(row['zip_img_path'])
                    with open(out_path, 'wb') as f:
                        f.write(img_data)
            except:
                continue

print("추출 완료!")

원천데이터_1.zip: 18653개 추출 중...


  0%|          | 0/18653 [00:00<?, ?it/s]

원천데이터_2.zip: 4980개 추출 중...


  0%|          | 0/4980 [00:00<?, ?it/s]

원천데이터_3.zip: 26367개 추출 중...


  0%|          | 0/26367 [00:00<?, ?it/s]

추출 완료!


In [ ]:
import os
EXTRACT_DIR = "/content/kfashion_images"
files = os.listdir(EXTRACT_DIR)
print(f"추출된 이미지 수: {len(files)}")

추출된 이미지 수: 50000


## zip에서 직접 읽는 Dataset

In [ ]:
# ================================================================
# 셀 5 — Dataset 정의
# ================================================================
import zipfile
import io
import os
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import torch

ZIP_ROOT = "/content/drive/MyDrive/codimodi/01_data/raw/aihub_kfashion"

# zip 파일 핸들 캐시
_zip_cache = {}

def get_zip(zip_name):
    if zip_name not in _zip_cache:
        _zip_cache[zip_name] = zipfile.ZipFile(
            os.path.join(ZIP_ROOT, zip_name), 'r'
        )
    return _zip_cache[zip_name]


class KFashionZipDataset(Dataset):
    """zip에서 직접 이미지 읽는 Dataset"""
    def __init__(self, df, transform=None):
        self.df        = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        try:
            zip_name = row["zip_name"]
            zip_path = row["zip_img_path"]
            zf       = get_zip(zip_name)
            img_data = zf.read(zip_path)
            img      = Image.open(io.BytesIO(img_data)).convert("RGB")
        except:
            img = Image.new("RGB", (224, 224), (200, 200, 200))

        if self.transform:
            img = self.transform(img)

        return (
            img,
            torch.tensor(int(row["h_style_id"]),    dtype=torch.long),
            torch.tensor(int(row["h_material_id"]), dtype=torch.long),
            torch.tensor(int(row["h_pattern_id"]),  dtype=torch.long),
        )


class KFashionLocalDataset(Dataset):
    """로컬 파일에서 읽는 Dataset (압축 해제 후 사용)"""
    def __init__(self, df, image_dir, transform=None):
        self.df        = df.reset_index(drop=True)
        self.image_dir = image_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        try:
            img_path = f"{self.image_dir}/{row['img_id']}.jpg"
            img      = Image.open(img_path).convert("RGB")
        except:
            img = Image.new("RGB", (224, 224), (200, 200, 200))

        if self.transform:
            img = self.transform(img)

        return (
            img,
            torch.tensor(int(row["h_style_id"]),    dtype=torch.long),
            torch.tensor(int(row["h_material_id"]), dtype=torch.long),
            torch.tensor(int(row["h_pattern_id"]),  dtype=torch.long),
        )


# 패턴 특화 augmentation
train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(20),
    transforms.ColorJitter(brightness=0.3, contrast=0.3,
                            saturation=0.3, hue=0.05),
    transforms.RandomGrayscale(p=0.1),
    transforms.RandomPerspective(distortion_scale=0.2, p=0.3),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                          [0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                          [0.229, 0.224, 0.225])
])

print("Dataset 클래스 정의 완료")

Dataset 클래스 정의 완료


## FashionHarmonyModel 로드 (속성 헤드만 학습)

In [ ]:
import os
import torch
import torch.nn as nn
import timm

SAVE_DIR = "/content/drive/MyDrive/fashion_harmony"
os.makedirs(SAVE_DIR, exist_ok=True)

# ===== 모델 정의 =====
class FashionBackbone(nn.Module):
    def __init__(self, embed_dim=512):
        super().__init__()
        base            = timm.create_model("efficientnet_b3", pretrained=True)
        self.features   = nn.Sequential(*list(base.children())[:-1])
        in_features     = base.classifier.in_features
        self.projection = nn.Sequential(
            nn.Linear(in_features, 1024),
            nn.BatchNorm1d(1024),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(1024, embed_dim),
        )

    def forward(self, x):
        feat = self.features(x).flatten(1)
        return self.projection(feat)


class AttributeHeads(nn.Module):
    def __init__(self, embed_dim=512):
        super().__init__()

        def _head(out_dim):
            return nn.Sequential(
                nn.Linear(embed_dim, 256),
                nn.BatchNorm1d(256),
                nn.ReLU(),
                nn.Dropout(0.3),
                nn.Linear(256, out_dim)
            )

        self.category_head = _head(len(CATEGORY_CLASSES))
        self.material_head = _head(len(MATERIAL_CLASSES))
        self.pattern_head  = _head(len(PATTERN_CLASSES))
        self.style_head    = _head(len(STYLE_CLASSES))

    def forward(self, emb):
        return {
            "category": self.category_head(emb),
            "material": self.material_head(emb),
            "pattern":  self.pattern_head(emb),
            "style":    self.style_head(emb),
        }


# 모델 초기화
backbone   = FashionBackbone(embed_dim=512).to(device)
attr_heads = AttributeHeads(embed_dim=512).to(device)

# 백본 로드 시도
print(f"SAVE_DIR 파일 목록: {os.listdir(SAVE_DIR)}")

backbone_candidates = [
    f"{SAVE_DIR}/backbone_cl.pt",
    f"{SAVE_DIR}/backbone_attr.pt",
]

loaded = False
for ckpt_path in backbone_candidates:
    if os.path.exists(ckpt_path):
        try:
            backbone.load_state_dict(torch.load(ckpt_path, map_location=device))
            print(f"백본 로드 완료: {ckpt_path}")
            loaded = True
            break
        except Exception as e:
            print(f"로드 실패 ({ckpt_path}): {e}")

if not loaded:
    print("백본 파일 없음 → ImageNet pretrained로 시작")

print("모델 준비 완료")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/49.3M [00:00<?, ?B/s]

SAVE_DIR 파일 목록: ['backbone_cl.pt', 'auto_labels.json', 'auto_labels_balanced.json', 'set_transformer.pt', 'model_st.pt', 'attr_proj.pt', 'attr_heads.pt', 'fashion_harmony_final.pt', 'attr_heads_kfashion.pt', 'backbone_attr.pt']
백본 로드 완료: /content/drive/MyDrive/fashion_harmony/backbone_cl.pt
모델 준비 완료


## 학습

In [ ]:
# 샘플링된 이미지만 압축 해제
import zipfile
import os
from tqdm.auto import tqdm

EXTRACT_DIR = "/content/kfashion_images"
os.makedirs(EXTRACT_DIR, exist_ok=True)

ZIP_ROOT = "/content/drive/MyDrive/codimodi/01_data/raw/aihub_kfashion"

# train_df + val_df에 있는 이미지만 추출
all_df = pd.concat([train_df, val_df], ignore_index=True)

zip_groups = all_df.groupby("zip_name")

for zip_name, group in zip_groups:
    zip_path = f"{ZIP_ROOT}/{zip_name}"
    print(f"{zip_name}: {len(group)}개 추출 중...")

    with zipfile.ZipFile(zip_path, 'r') as zf:
        for _, row in tqdm(group.iterrows(), total=len(group)):
            try:
                img_path = row["zip_img_path"]
                out_path = f"{EXTRACT_DIR}/{row['img_id']}.jpg"
                if not os.path.exists(out_path):
                    img_data = zf.read(img_path)
                    with open(out_path, 'wb') as f:
                        f.write(img_data)
            except:
                continue

print("추출 완료!")

원천데이터_1.zip: 18653개 추출 중...


  0%|          | 0/18653 [00:00<?, ?it/s]

원천데이터_2.zip: 4980개 추출 중...


  0%|          | 0/4980 [00:00<?, ?it/s]

원천데이터_3.zip: 26367개 추출 중...


  0%|          | 0/26367 [00:00<?, ?it/s]

추출 완료!


In [ ]:
# ================================================================
# 셀 7 — 학습
# ================================================================
import torch.nn as nn
from tqdm.auto import tqdm

EXTRACT_DIR = "/content/kfashion_images"

train_ds = KFashionLocalDataset(train_df, EXTRACT_DIR, train_transform)
val_ds   = KFashionLocalDataset(val_df,   EXTRACT_DIR, val_transform)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True,
                           num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=32, shuffle=False,
                           num_workers=2, pin_memory=True)

print(f"train: {len(train_ds)}, val: {len(val_ds)}")

# Loss
ce_style    = nn.CrossEntropyLoss()
ce_material = nn.CrossEntropyLoss(ignore_index=-1)
ce_pattern  = nn.CrossEntropyLoss(ignore_index=-1)

# Optimizer
optimizer = torch.optim.AdamW([
    {"params": backbone.parameters(),   "lr": 1e-5},
    {"params": attr_heads.parameters(), "lr": 1e-4},
], weight_decay=1e-4)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=20)

EPOCHS   = 20
best_acc = 0

for epoch in range(EPOCHS):
    backbone.train()
    attr_heads.train()

    total_loss = 0
    correct    = {"style": 0, "material": 0, "pattern": 0}
    total      = {"style": 0, "material": 0, "pattern": 0}

    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}")
    for img, y_style, y_material, y_pattern in pbar:
        img        = img.to(device)
        y_style    = y_style.to(device)
        y_material = y_material.to(device)
        y_pattern  = y_pattern.to(device)

        emb   = backbone(img)
        preds = attr_heads(emb)

        loss_style    = ce_style(preds["style"], y_style)
        loss_material = ce_material(preds["material"], y_material)
        loss_pattern  = ce_pattern(preds["pattern"],   y_pattern)

        loss = loss_style + loss_material + loss_pattern

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(
            list(backbone.parameters()) + list(attr_heads.parameters()), 1.0
        )
        optimizer.step()

        correct["style"] += (preds["style"].argmax(1) == y_style).sum().item()
        total["style"]   += y_style.size(0)

        mat_mask = y_material != -1
        if mat_mask.sum() > 0:
            correct["material"] += (preds["material"].argmax(1)[mat_mask] == y_material[mat_mask]).sum().item()
            total["material"]   += mat_mask.sum().item()

        pat_mask = y_pattern != -1
        if pat_mask.sum() > 0:
            correct["pattern"] += (preds["pattern"].argmax(1)[pat_mask] == y_pattern[pat_mask]).sum().item()
            total["pattern"]   += pat_mask.sum().item()

        total_loss += loss.item()
        pbar.set_postfix({
            "loss": f"{loss.item():.4f}",
            "sty":  f"{correct['style']/max(total['style'],1):.3f}",
            "mat":  f"{correct['material']/max(total['material'],1):.3f}",
            "pat":  f"{correct['pattern']/max(total['pattern'],1):.3f}",
        })

    scheduler.step()

    # Val 평가
    backbone.eval()
    attr_heads.eval()
    val_correct = {"style": 0, "material": 0, "pattern": 0}
    val_total   = {"style": 0, "material": 0, "pattern": 0}

    with torch.no_grad():
        for img, y_style, y_material, y_pattern in val_loader:
            img        = img.to(device)
            y_style    = y_style.to(device)
            y_material = y_material.to(device)
            y_pattern  = y_pattern.to(device)

            emb   = backbone(img)
            preds = attr_heads(emb)

            val_correct["style"] += (preds["style"].argmax(1) == y_style).sum().item()
            val_total["style"]   += y_style.size(0)

            mat_mask = y_material != -1
            if mat_mask.sum() > 0:
                val_correct["material"] += (preds["material"].argmax(1)[mat_mask] == y_material[mat_mask]).sum().item()
                val_total["material"]   += mat_mask.sum().item()

            pat_mask = y_pattern != -1
            if pat_mask.sum() > 0:
                val_correct["pattern"] += (preds["pattern"].argmax(1)[pat_mask] == y_pattern[pat_mask]).sum().item()
                val_total["pattern"]   += pat_mask.sum().item()

    sty_acc = val_correct["style"]    / max(val_total["style"], 1)
    mat_acc = val_correct["material"] / max(val_total["material"], 1)
    pat_acc = val_correct["pattern"]  / max(val_total["pattern"], 1)
    avg_acc = (sty_acc + mat_acc + pat_acc) / 3

    print(f"Epoch {epoch+1}: "
          f"style={sty_acc:.4f} | "
          f"material={mat_acc:.4f} | "
          f"pattern={pat_acc:.4f} | "
          f"avg={avg_acc:.4f}")

    if avg_acc > best_acc:
        best_acc = avg_acc
        torch.save(backbone.state_dict(),   f"{SAVE_DIR}/backbone_attr.pt")
        torch.save(attr_heads.state_dict(), f"{SAVE_DIR}/attr_heads_kfashion.pt")
        print(f"  → 저장 완료 (avg_acc={avg_acc:.4f})")

print(f"\n학습 완료! 최고 avg_acc={best_acc:.4f}")

train: 42500, val: 7500


Epoch 1/20:   0%|          | 0/1329 [00:00<?, ?it/s]

Epoch 1: style=0.2801 | material=0.5175 | pattern=0.7266 | avg=0.5081
  → 저장 완료 (avg_acc=0.5081)


Epoch 2/20:   0%|          | 0/1329 [00:00<?, ?it/s]

Epoch 2: style=0.2924 | material=0.5400 | pattern=0.7529 | avg=0.5284
  → 저장 완료 (avg_acc=0.5284)


...

Epoch 18: style=0.4059 | material=0.6353 | pattern=0.8090 | avg=0.6167
  → 저장 완료 (avg_acc=0.6167)


Epoch 19/20:   0%|          | 0/1329 [00:00<?, ?it/s]

Epoch 19: style=0.4025 | material=0.6373 | pattern=0.8072 | avg=0.6157


Epoch 20/20:   0%|          | 0/1329 [00:00<?, ?it/s]

Epoch 20: style=0.4011 | material=0.6360 | pattern=0.8087 | avg=0.6153

학습 완료! 최고 avg_acc=0.6167
